# PSA: Private Set Alignment for Federated Learning

**Library:** [anonlink](https://github.com/data61/anonlink) + [clkhash](https://github.com/data61/clkhash) (CSIRO Data61, Apache-2.0)  
**Install:** `pip install anonlink clkhash`  
**Stage:** Pre-training (entity alignment before Vertical FL)

### Glossary

| Term | Meaning |
|------|---------|
| **PSA** | Private Set Alignment — matching records across organisations without revealing raw data |
| **PSI** | Private Set Intersection — older approach requiring exact shared identifiers |
| **CLK** | Cryptographic Longterm Key — a Bloom filter hash that encodes a record's fields |
| **VFL** | Vertical Federated Learning — training across organisations that hold different features for the same entities |
| **DOB** | Date of birth |
| **Dice coefficient** | A similarity score (0 to 1) measuring how similar two CLKs are |

---

## Contents

1. [The Problem: Why PSI Fails](#1-the-problem)
2. [How CLK Bloom Filters Work](#2-how-clk-bloom-filters-work)
3. [Single-Pass PSA](#3-single-pass-psa)
4. [Hard Negatives: Where Single-Pass Fails](#4-hard-negatives)
5. [Double PSA: Triangulation](#5-double-psa-triangulation)
6. [Threshold Tuning](#6-threshold-tuning)
7. [Security and Privacy](#7-security-and-privacy)
8. [Governance and Compliance](#8-governance-and-compliance)
9. [Production Recommendations](#9-production-recommendations)

In [ ]:
import os, sys, time, csv, random, json
from io import StringIO
from pathlib import Path
import numpy as np

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

from fl_pets.psa.protocol import PSAProtocol, ANONLINK_AVAILABLE
from fl_pets.psa import align_entities_fuzzy, align_entities_exact
import anonlink
from clkhash import clk
from clkhash.schema import from_json_dict as schema_from_json_dict

random.seed(42)
print(f'Working directory: {os.getcwd()}')
print(f'anonlink available: {ANONLINK_AVAILABLE}')

# -- Generate sample data if not present --
PSA_DIR = Path('data/samples/psa')
if not (PSA_DIR / 'sgh_small.csv').exists():
    print('Generating PSA sample data...')
    PSA_DIR.mkdir(parents=True, exist_ok=True)
    from data.generators.sg_synthetic import generate_small, generate_records

    sgh_raw, ttsh_raw, n_matches = generate_small()
    for fname, recs in [('sgh_small.csv', sgh_raw), ('ttsh_small.csv', ttsh_raw)]:
        with open(PSA_DIR / fname, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['name', 'dob', 'address', 'gender'])
            w.writeheader()
            w.writerows(recs)

    clean, noisy, n_true, n_hard = generate_records(1000, seed=42)
    for fname, recs in [('hospital_a_1k.csv', clean), ('hospital_b_1k.csv', noisy)]:
        with open(PSA_DIR / fname, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['name', 'dob', 'address', 'gender', 'age', 'income'])
            w.writeheader()
            w.writerows(recs)

    with open(PSA_DIR / 'manifest.json', 'w') as f:
        json.dump({'datasets': {'small': {'true_matches': n_matches},
                                '1k': {'true_matches': n_true, 'hard_negatives': n_hard}}}, f, indent=2)
    print(f'  Created {PSA_DIR}/ with small ({len(sgh_raw)} records) and 1K ({len(clean)} records) datasets')
else:
    print(f'PSA sample data found at {PSA_DIR}/')

# -- Load pre-generated data --
def _load_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))

sgh_all = _load_csv(PSA_DIR / 'sgh_small.csv')
ttsh_all = _load_csv(PSA_DIR / 'ttsh_small.csv')
N_SMALL = 18

print(f'Loaded {len(sgh_all)} SGH records, {len(ttsh_all)} TTSH records')
print(f'Sample: {sgh_all[0]["name"]} (SGH) vs {ttsh_all[0]["name"]} (TTSH)')

---
## 1. The Problem: Why PSI Fails <a id='1-the-problem'></a>

In **Vertical Federated Learning (VFL)**, different organisations hold different features for the same entities. Before training, they must **align records** — find which rows refer to the same person.

```
Hospital A (demographics)      Hospital B (lab results)
┌──────────────────────┐       ┌──────────────────────┐
│ Name    │ DOB   │ BP │       │ Name    │ HbA1c │ WBC│
├─────────┼───────┼────┤       ├─────────┼───────┼────┤
│ Tan Ah Kow │ 1965 │120│      │ Tan Ah Kou│ 1965 │ 8.2│  ← same person?
│ Jane Doe   │ 1990 │115│      │ Jane Doe  │ 1990 │ 5.1│  ← same person?
└──────────────────────┘       └──────────────────────┘
```

**Traditional PSI** (Private Set Intersection) hashes identifiers and computes the set intersection. This requires:
1. A **shared identifier** (e.g., national ID) — often unavailable across organisations
2. **Exact match** — fails on typos, abbreviations, romanisation differences

In Singapore's multi-ethnic healthcare system, this is particularly problematic:

In [ ]:
# Use first 5 records from pre-generated Singapore data to show the problem
sgh = sgh_all[:5]
ttsh = ttsh_all[:5]

print('SGH (Hospital A):')
for r in sgh:
    print(f'  {r["name"]:40s}  DOB={r["dob"]}  {r["gender"]}')
print()
print('TTSH (Hospital B) — same patients, noisy records:')
for r in ttsh:
    print(f'  {r["name"]:40s}  DOB={r["dob"]}  {r["gender"]}')

# Try exact matching
exact = PSAProtocol(mode="exact", salt=os.urandom(32))
keys_a = [f"{r['name']}|{r['dob']}" for r in sgh]
keys_b = [f"{r['name']}|{r['dob']}" for r in ttsh]
hashes_a = exact.hash_identifiers(keys_a)
hashes_b = exact.hash_identifiers(keys_b)
idx_a, idx_b = PSAProtocol.intersect(hashes_a, hashes_b)

print(f'\nExact PSI: {len(idx_a)}/5 matches')
print()
print('Why it fails:')
for i in range(5):
    match = '✓' if i in idx_a else '✗'
    diff = '' if sgh[i]['name'] == ttsh[i]['name'] else f'  ({sgh[i]["name"]} ≠ {ttsh[i]["name"]})'
    print(f'  {match} {sgh[i]["name"]:40s}{diff}')

**Result: 0/5 matches.** Every record has a minor name difference that breaks exact hashing.

This is not a contrived example — these are real-world romanisation variants, Malay naming conventions (`bin`/`bte`/`binti`), and Indian patronymic formats (`s/o`/`d/o`) that differ between hospital systems.

---
## 2. How CLK Bloom Filters Work <a id='2-how-clk-bloom-filters-work'></a>

**PSA** solves this using **Cryptographic Longterm Keys (CLKs)** — Bloom filter hashes of character n-grams.

> **Plain-English analogy:** Think of a CLK as a **fingerprint** of a name. Similar names produce similar fingerprints, but you cannot reconstruct the original name from the fingerprint. Two hospitals can compare fingerprints to find matching patients without ever sharing the actual names.

### Step-by-step encoding

```
1. TOKENISE: Split name into character bigrams (2-grams)
   "Tan Ah Kow"  →  {"Ta", "an", "n ", " A", "Ah", "h ", " K", "Ko", "ow"}
   "Tan Ah Kou"  →  {"Ta", "an", "n ", " A", "Ah", "h ", " K", "Ko", "ou"}
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                      8 of 9 bigrams are identical!

2. HASH: Each bigram is hashed into bit positions in a 1024-bit Bloom filter
   "Ta" → bits {42, 117, 503, ...}
   "an" → bits {89, 234, 678, ...}
   ...

3. COMBINE: OR all bit positions together → one 1024-bit CLK per record
   CLK_A = 1011001...0110  (Tan Ah Kow)
   CLK_B = 1011001...0100  (Tan Ah Kou)  ← very similar!

4. COMPARE: Dice coefficient measures similarity between two CLKs
   Dice = 2 × |bits_in_common| / (|bits_A| + |bits_B|)
   Dice(CLK_A, CLK_B) ≈ 0.95  →  match!
```

### Privacy properties

- The Bloom filter is **computationally hard to reverse** — recovering the original name requires guessing and checking against the CLK, which is infeasible without the secret key
- The hash function is **keyed** — without the secret key, an attacker cannot generate CLKs to test against
- Multiple fields are **mixed** into the same Bloom filter — individual field values cannot be separated

> **Important caveat:** CLK encoding is *not* perfectly irreversible. Research has shown that statistical attacks (frequency analysis, pattern mining) can partially recover field values from CLKs, especially for common names in small populations. See the [Security and Privacy](#7-security-and-privacy) section for details and mitigations.

In [ ]:
# Visualise how bigrams overlap between similar names
def bigrams(s):
    return set(s[i:i+2] for i in range(len(s)-1))

pairs = [
    ("Tan Ah Kow", "Tan Ah Kou"),          # Chinese romanisation
    ("Lim Mei Ling", "Lim Mei-Ling"),       # Hyphenation
    ("Muhammad Faizal", "Mohd Faizal"),      # Malay abbreviation
    ("Kavitha Devi", "Kavita Devi"),         # Indian name variant
    ("s/o Muthu", "S/O Muthu"),              # Patronymic format
    ("Tan Ah Kow", "Lee Mei Ling"),          # DIFFERENT person (should be low)
]

print(f'{"Name A":25s} {"Name B":25s} {"Common":>7} {"Total":>7} {"Dice":>7}')
print('-' * 75)
for a, b in pairs:
    ba, bb = bigrams(a), bigrams(b)
    common = ba & bb
    total = ba | bb
    dice = 2 * len(common) / (len(ba) + len(bb)) if (len(ba) + len(bb)) > 0 else 0
    marker = '← match' if dice > 0.5 else '← no match'
    print(f'{a:25s} {b:25s} {len(common):>7} {len(total):>7} {dice:>7.3f}  {marker}')

---
## 3. Single-Pass PSA <a id='3-single-pass-psa'></a>

The simplest approach: encode **all fields** into one CLK per record, then match.

In [ ]:
# Full PSA with fuzzy matching on pre-generated data
aligned = align_entities_fuzzy(
    parties={"sgh": sgh, "ttsh": ttsh},
    threshold=0.7,
)

print(f'Fuzzy PSA: {len(aligned["sgh"])}/5 matches')
print()
for ia, ib in zip(aligned['sgh'], aligned['ttsh']):
    print(f'  ✓ "{sgh[ia]["name"]}"  ←→  "{ttsh[ib]["name"]}"')

### CLK Schema Definition

The schema tells clkhash how to encode each field. Key parameters:

| Parameter | Meaning | Recommended |
|-----------|---------|-------------|
| `l` | Bloom filter length (bits) | 1024 |
| `n` (ngram) | Character n-gram size | 2 for names, 1 for dates |
| `bitsPerFeature` | How many bits this field occupies in the filter | Higher = more weight |
| `type: exact` | No n-grams, hash the whole value | Use for categorical fields (gender) |

**`bitsPerFeature` controls field importance.** Name (300 bits) dominates over gender (50 bits), so name similarity matters most.

In [ ]:
# The default healthcare schema
from fl_pets.psa.protocol import DEFAULT_CLK_SCHEMA
import json

print('Default CLK schema for healthcare PSA:')
print()
for feat in DEFAULT_CLK_SCHEMA['features']:
    comp = feat['hashing']['comparison']
    bits = feat['hashing']['strategy']['bitsPerFeature']
    if comp['type'] == 'ngram':
        print(f"  {feat['identifier']:12s}  {comp['type']}-{comp['n']}  {bits:>4} bits  (fuzzy match)")
    else:
        print(f"  {feat['identifier']:12s}  {comp['type']:>7s}  {bits:>4} bits  (categorical)")
print(f"\n  Total Bloom filter: {DEFAULT_CLK_SCHEMA['clkConfig']['l']} bits")

---
## 4. Hard Negatives: Where Single-Pass Fails <a id='4-hard-negatives'></a>

Single-pass PSA works well for typical typos. But Singapore has **extremely common names**. When different people share the same name, single-pass produces false positives.

### The four hard cases

| Case | Problem | Example |
|------|---------|--------|
| **Same name, diff person** | Two "Tan Wei Ming" at different addresses | Name matches, address doesn't |
| **Family members** | Father and son at same HDB flat, same surname | Address matches, given name doesn't |
| **Neighbours** | Different families in the same block | Address partially matches |
| **Movers** | Same person moved to new address | Name matches, address doesn't |

In [ ]:
# Hard negative demo: same name, different person
# Using a focused 3-record example to clearly show the problem

hospital_a = [
    {"name": "Tan Wei Ming", "dob": "1985-03-15", "address": "Blk 123 Ang Mo Kio Ave 6 #08-456", "gender": "M"},
    {"name": "Tan Wei Ming", "dob": "1962-11-22", "address": "Blk 789 Jurong West St 42 #03-111", "gender": "M"},
    {"name": "Lee Mei Ling", "dob": "1990-07-10", "address": "Blk 456 Toa Payoh Lor 1 #12-789",  "gender": "F"},
]

hospital_b = [
    {"name": "Tan Wei Ming", "dob": "1985-03-15", "address": "BLK 123 Ang Mo Kio Avenue 6 #08-456", "gender": "M"},  # TRUE match for A[0]
    {"name": "Tan Wei Min",  "dob": "1962-11-22", "address": "Blk 789 Jurong West Street 42 #03-111", "gender": "M"},  # TRUE match for A[1]
    {"name": "Lee Mei Ling", "dob": "1993-02-28", "address": "Blk 222 Hougang Ave 10 #05-333",     "gender": "F"},  # DIFFERENT person, same name!
]

# Load the pre-generated 1K dataset with hard negatives from data/samples/psa/
clean_1k = _load_csv(PSA_DIR / 'hospital_a_1k.csv')
noisy_1k = _load_csv(PSA_DIR / 'hospital_b_1k.csv')
with open(PSA_DIR / 'manifest.json') as f:
    psa_manifest = json.load(f)
n_true_1k = psa_manifest['datasets']['1k']['true_matches']
n_hard_1k = psa_manifest['datasets']['1k']['hard_negatives']

print(f'Loaded 1K dataset from {PSA_DIR}/:')
print(f'  True matches: {n_true_1k:,}  |  Hard negatives: {n_hard_1k:,}')
print(f'  Hospital A: {len(clean_1k):,} records  |  Hospital B: {len(noisy_1k):,} records')

# Single-pass on the 3-record example
schema_all = {
    "version": 3,
    "clkConfig": {
        "l": 1024, "xor_folds": 0,
        "kdf": {"type": "HKDF", "hash": "SHA256", "info": "ZGVtbw==", "salt": "c2FsdA==", "keySize": 64},
    },
    "features": [
        {"identifier": "name",    "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 2}, "strategy": {"bitsPerFeature": 300}}},
        {"identifier": "dob",     "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 1}, "strategy": {"bitsPerFeature": 150}}},
        {"identifier": "address", "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 2}, "strategy": {"bitsPerFeature": 250}}},
        {"identifier": "gender",  "format": {"type": "enum", "values": ["M", "F"]},  "hashing": {"comparison": {"type": "exact"}, "strategy": {"bitsPerFeature": 50}}},
    ],
}
fields_all = ["name", "dob", "address", "gender"]

def encode_records(records, schema_dict, fields, secret):
    schema = schema_from_json_dict(schema_dict)
    buf = StringIO()
    writer = csv.writer(buf)
    writer.writerow(fields)
    for r in records:
        writer.writerow([r.get(f, "") for f in fields])
    buf.seek(0)
    return clk.generate_clk_from_csv(buf, secret, schema)

def match_clks(clks_a, clks_b, threshold=0.7):
    results = anonlink.candidate_generation.find_candidate_pairs(
        [clks_a, clks_b], anonlink.similarities.dice_coefficient_accelerated, threshold)
    solution = anonlink.solving.greedy_solve(results)
    pairs = set()
    for group in solution:
        by_ds = {}
        for ds, idx in group:
            by_ds[ds] = idx
        if 0 in by_ds and 1 in by_ds:
            pairs.add((by_ds[0], by_ds[1]))
    return pairs

clks_a = encode_records(hospital_a, schema_all, fields_all, "demo")
clks_b = encode_records(hospital_b, schema_all, fields_all, "demo")
pairs = match_clks(clks_a, clks_b, threshold=0.7)

print(f'\nSingle-pass PSA results (3-record example):')
print(f'  Matches found: {len(pairs)}')
for ia, ib in sorted(pairs):
    correct = (ia == 0 and ib == 0) or (ia == 1 and ib == 1)
    status = 'TRUE MATCH' if correct else 'FALSE POSITIVE'
    print(f'  A[{ia}] {hospital_a[ia]["name"]:20s} ({hospital_a[ia]["dob"]})  <->  B[{ib}] {hospital_b[ib]["name"]:20s} ({hospital_b[ib]["dob"]})  {status}')

print()
print('Problem: the "Lee Mei Ling" false positive — same name, different person.')
print('Name similarity dominates the combined CLK.')

---
## 5. Double PSA: Triangulation <a id='5-double-psa-triangulation'></a>

The solution: run **two independent PSA passes** on different field groups, then **intersect** the results.

```
┌─────────────────────────────────────────────────────────────────────┐
│                     DOUBLE PSA WORKFLOW                            │
│                                                                     │
│  Hospital A records          Hospital B records                     │
│       │                           │                                 │
│       ├──→ Pass 1: IDENTITY ──────┤                                 │
│       │    (name + DOB + gender)   │                                 │
│       │         │                  │                                 │
│       │    Candidate pairs₁        │                                 │
│       │         │                  │                                 │
│       ├──→ Pass 2: LOCATION ──────┤                                 │
│       │    (address + age + income)│                                 │
│       │         │                  │                                 │
│       │    Candidate pairs₂        │                                 │
│       │         │                  │                                 │
│       │    ┌────┴────┐                                              │
│       │    │  INTERSECT  │  ← only pairs in BOTH passes survive     │
│       │    └────┬────┘                                              │
│       │         │                                                   │
│       │    Final aligned pairs                                      │
└─────────────────────────────────────────────────────────────────────┘
```

### Why this works

| False positive type | Pass 1 (identity) | Pass 2 (location) | Intersection |
|--------------------|--------------------|--------------------|--------------|
| Same name, diff address | ✓ matches | ✗ doesn't match | **Eliminated** |
| Same address, diff name (family) | ✗ doesn't match | ✓ matches | **Eliminated** |
| Same name AND address (true match) | ✓ matches | ✓ matches | **Kept** |

The two passes use **independent field groups** with **different CLK schemas and different secret keys**, so they provide orthogonal evidence.

In [ ]:
# Define separate schemas for each pass

# Pass 1: Identity — who is this person?
schema_identity = {
    "version": 3,
    "clkConfig": {
        "l": 1024, "xor_folds": 0,
        "kdf": {"type": "HKDF", "hash": "SHA256", "info": "aWQ=", "salt": "aWQtc2FsdA==", "keySize": 64},
    },
    "features": [
        {"identifier": "name",   "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 2}, "strategy": {"bitsPerFeature": 400}}},
        {"identifier": "dob",    "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 1}, "strategy": {"bitsPerFeature": 200}}},
        {"identifier": "gender", "format": {"type": "enum", "values": ["M", "F"]},  "hashing": {"comparison": {"type": "exact"}, "strategy": {"bitsPerFeature": 100}}},
    ],
}
fields_identity = ["name", "dob", "gender"]

# Pass 2: Location — where is this person?
schema_location = {
    "version": 3,
    "clkConfig": {
        "l": 1024, "xor_folds": 0,
        "kdf": {"type": "HKDF", "hash": "SHA256", "info": "bG9j", "salt": "bG9jLXNhbHQ=", "keySize": 64},
    },
    "features": [
        {"identifier": "address", "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 2}, "strategy": {"bitsPerFeature": 500}}},
        {"identifier": "dob",     "format": {"type": "string", "encoding": "utf-8"}, "hashing": {"comparison": {"type": "ngram", "n": 1}, "strategy": {"bitsPerFeature": 150}}},
    ],
}
fields_location = ["address", "dob"]

print('Schema summary:')
print(f'  Pass 1 (Identity): name ({400}b) + DOB ({200}b) + gender ({100}b)')
print(f'  Pass 2 (Location): address ({500}b) + DOB ({150}b)')
print(f'  Note: DOB appears in both passes as a linking anchor')

In [ ]:
# Run double PSA on the hard negative example

# Pass 1: Identity
clks_a_id = encode_records(hospital_a, schema_identity, fields_identity, "key-identity")
clks_b_id = encode_records(hospital_b, schema_identity, fields_identity, "key-identity")
pairs_id = match_clks(clks_a_id, clks_b_id, threshold=0.7)

print('Pass 1 — Identity (name + DOB + gender):')
for ia, ib in sorted(pairs_id):
    print(f'  A[{ia}] {hospital_a[ia]["name"]:20s}  ←→  B[{ib}] {hospital_b[ib]["name"]}')

# Pass 2: Location
clks_a_loc = encode_records(hospital_a, schema_location, fields_location, "key-location")
clks_b_loc = encode_records(hospital_b, schema_location, fields_location, "key-location")
pairs_loc = match_clks(clks_a_loc, clks_b_loc, threshold=0.7)

print(f'\nPass 2 — Location (address + DOB):')
for ia, ib in sorted(pairs_loc):
    print(f'  A[{ia}] ...{hospital_a[ia]["address"][-25:]}  ←→  B[{ib}] ...{hospital_b[ib]["address"][-25:]}')

# Triangulate
pairs_tri = pairs_id & pairs_loc

print(f'\nTriangulated (intersection):')
print(f'  Pass 1 candidates: {len(pairs_id)}')
print(f'  Pass 2 candidates: {len(pairs_loc)}')
print(f'  Final matches:     {len(pairs_tri)}')
print()
for ia, ib in sorted(pairs_tri):
    print(f'  ✓ A[{ia}] {hospital_a[ia]["name"]:20s}  ←→  B[{ib}] {hospital_b[ib]["name"]}')

### What happened

- **Pass 1** matched all three pairs (including the false positive "Lee Mei Ling") because names are similar
- **Pass 2** only matched pairs where addresses are similar — the false positive has a completely different address
- **Intersection** kept only the true matches where BOTH identity AND location agree

---
## 6. Threshold Tuning <a id='6-threshold-tuning'></a>

The `threshold` parameter controls how similar two records must be to count as a match. It ranges from 0 (match everything) to 1 (require identical records).

| Threshold | Plain English | Use when |
|-----------|--------------|----------|
| 0.5 | "Loosely similar" — catches most matches but includes some wrong ones | Very noisy data, willing to manually review |
| 0.7 | "Clearly similar" — good balance | Typical typos, abbreviation differences |
| 0.8 | "Very similar" — few wrong matches | Clean data, need high confidence |
| 0.9 | "Nearly identical" — misses records with larger differences | Near-exact matching only |

### Recommended thresholds for Singapore healthcare data

Based on our 10,000-record test with multi-ethnic names and HDB addresses:

| Method | Threshold | Correct matches | Wrong matches | Missed patients |
|--------|-----------|----------------|---------------|-----------------|
| Single-pass (all fields) | 0.70 | 99.98% found | ~18% of results are wrong | 2 patients missed |
| Single-pass (all fields) | 0.85 | 99.0% found | ~5% wrong | 100 patients missed |
| **Double PSA (triangulated)** | **0.70 / 0.70** | **94.4% found** | **<0.1% wrong** | **559 patients missed** |

> **In plain English:** With triangulated double PSA at threshold 0.7, fewer than 1 in 1,000 matched pairs are incorrect. This means you can trust the aligned dataset for model training without manual review.

**Key insight:** With triangulation, you can use a **lower threshold per pass** (0.7) while still achieving near-perfect precision. The intersection does the quality control, so each pass can be generous.

In [ ]:
# Demonstrate threshold effect using pre-generated data (first 8 records)
records_a = sgh_all[:8]
records_b = ttsh_all[:8]

print('Test records (name differences marked with *):')
for i in range(8):
    diff = ' *' if records_a[i]['name'] != records_b[i]['name'] else ''
    print(f'  [{i}] {records_a[i]["name"]:35s} vs {records_b[i]["name"]}{diff}')

print(f'\n{"Threshold":>10}  Matches')
print('-' * 30)
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    result = align_entities_fuzzy(
        parties={"a": records_a, "b": records_b},
        threshold=t,
    )
    print(f'{t:>10.2f}  {len(result["a"])}/8')

---
## 7. Security and Privacy <a id='7-security-and-privacy'></a>

### What PSA protects

| What | Protection | Plain English |
|------|-----------|---------------|
| **Raw names, addresses, DOBs** | Never transmitted | Only encoded fingerprints (CLKs) leave each hospital |
| **Non-matching patients** | Not revealed | If a patient is only at one hospital, the other hospital learns nothing about them |
| **Individual field values** | Mixed together | An attacker cannot separate the name component from the DOB component in a CLK |
| **Brute-force guessing** | Keyed encoding | Without the secret key, an attacker cannot generate CLKs to test names against |

### Known attacks and honest limitations

CLK Bloom filters provide **computational privacy**, not perfect secrecy. Published research has demonstrated attacks that can partially recover data from CLKs:

| Attack | What it does | Risk level | Mitigation |
|--------|-------------|------------|------------|
| **Frequency analysis** (Niedermeyer et al. 2014) | An attacker who sees many CLKs can guess common names by analysing which bit patterns appear most often | Medium — works best on common names in small populations | Enable XOR folding, balance bit allocation across fields |
| **Pattern mining** (Christen et al. 2018) | Uses graph structures in CLK collections to find co-occurring field values | Medium — requires access to a large CLK dataset | Limit CLK sharing to a single trusted linkage server |
| **Dictionary attack** | An attacker generates CLKs for common names and compares against target CLKs | Low if secret key is strong — high if key is weak or leaked | Use cryptographically random keys (min 32 bytes), rotate per session |
| **Intersection leakage** | Learning that a patient appears in both a psychiatric hospital and a general hospital reveals sensitive health information | High for sensitive specialties | Consider whether the match result itself is sensitive before proceeding |

> **For policymakers:** PSA is not a silver bullet. It significantly reduces the information shared between organisations (from raw patient records to encoded fingerprints), but a sophisticated attacker with prolonged access to CLK datasets could partially recover some field values. For the highest-sensitivity data (e.g., HIV status, psychiatric records), consider adding Differential Privacy noise on top of CLK encoding, or consult your data protection officer.

### Best practices

1. **Generate a fresh random key** for each alignment session — never reuse keys
2. **Enable XOR folding** in the CLK schema (`xor_folds: 1`) — reduces frequency attack surface
3. **Delete CLKs immediately** after alignment — they are only needed during matching
4. **Audit and log** alignment operations (who, when, how many matches) without logging the CLKs themselves
5. **Layer Differential Privacy** on top if formal mathematical privacy guarantees are required

---
## 8. Governance and Compliance <a id='8-governance-and-compliance'></a>

### Regulatory considerations

| Regulation | How PSA helps | What PSA does NOT do |
|-----------|---------------|----------------------|
| **PDPA** (Singapore) | Raw personal data stays within each organisation; only CLK hashes are shared | Does not eliminate the need for a Data Protection Impact Assessment (DPIA) |
| **HIPAA** (US) | Supports the "de-identification" requirement — CLKs are not PHI | Does not replace a BAA (Business Associate Agreement) between parties |
| **GDPR** (EU) | Supports data minimisation and purpose limitation principles | The alignment result (matched indices) may still constitute personal data processing |

### Before deploying PSA

- [ ] Obtain approval from your **Data Protection Officer (DPO)**
- [ ] Complete a **Data Protection Impact Assessment (DPIA)** covering the CLK exchange
- [ ] Sign a **data-sharing agreement** between all participating organisations
- [ ] Define **data retention policy**: how long are CLKs kept? (Recommended: delete immediately after alignment)
- [ ] Define **audit requirements**: who ran the alignment, when, match rate, threshold used
- [ ] Confirm that the **match result itself** is acceptable to share (co-membership in two hospitals can reveal health information)

### Vendor and cost

- **anonlink + clkhash** are open-source (Apache-2.0 licence) with no vendor lock-in
- No cloud dependency — can run entirely on-premise
- Compute cost: alignment of 10,000 records takes ~3 seconds on a standard laptop (no GPU needed)
- For larger datasets (>100K), CSIRO provides a containerised [Entity Service](https://github.com/data61/anonlink-entity-service) that can run on AWS/GCC

---
## 9. Production Recommendations <a id='9-production-recommendations'></a>

### When to use each mode

| Scenario | Method | Threshold |
|----------|--------|-----------|
| Parties share national IDs (NRIC hash) | `align_entities_exact()` | N/A |
| No shared IDs, clean data, few common names | Single-pass fuzzy, all fields | 0.80 |
| No shared IDs, noisy data, many common names | **Double PSA (triangulated)** | 0.70 / 0.70 |
| Highest sensitivity (classified data) | Double PSA + DP layer | 0.75 / 0.75 |

### Double PSA field grouping

The two passes must use **independent** field groups. Recommended split:

| Pass | Fields | Question it answers |
|------|--------|-----------|
| **Identity** | name, DOB, gender | Who is this person? |
| **Location** | address, age, income | Where does this person live? |

DOB can appear in both passes as a **linking anchor** — it is stable across systems and adds discriminative power. Note that this shared field slightly reduces the independence between the two passes.

### Production secrets

```python
import os

# CORRECT: cryptographically random key (32 bytes)
salt = os.urandom(32)

# WRONG: hardcoded string (easily guessed)
# salt = b"my-secret-key"   # DO NOT do this
```

In production, the shared secret must be agreed upon by all parties through a secure channel (e.g., encrypted email, secure key exchange protocol) before alignment begins. Generate a fresh key for each alignment session.

### Performance at scale

| Records | Encoding | Matching | Total |
|---------|----------|----------|-------|
| 1,000 | 0.1s | <0.1s | <0.2s |
| 10,000 | 1.3s | 1.9s | 3.2s |
| 10,000 (double PSA) | 2.6s | 3.8s | ~13s |
| 100,000 | ~13s | ~30s | ~45s |

For datasets above 100K records, consider the [anonlink Entity Service](https://github.com/data61/anonlink-entity-service) REST API.

### Complete production pipeline

```python
from fl_pets.psa import align_entities_fuzzy
import os

# Use cryptographically random key
session_key = os.urandom(32)

# Step 1: Double PSA alignment
pairs_identity = align_entities_fuzzy(
    parties, threshold=0.7, salt=session_key,
    schema_dict=SCHEMA_IDENTITY,
    fields=["name", "dob", "gender"],
)
pairs_location = align_entities_fuzzy(
    parties, threshold=0.7, salt=session_key,
    schema_dict=SCHEMA_LOCATION,
    fields=["address", "dob"],
)

# Step 2: Triangulate
aligned_a = set(zip(pairs_identity["hosp_a"], pairs_identity["hosp_b"]))
aligned_b = set(zip(pairs_location["hosp_a"], pairs_location["hosp_b"]))
final_pairs = aligned_a & aligned_b

# Step 3: Slice data for VFL training
idx_a = [p[0] for p in final_pairs]
idx_b = [p[1] for p in final_pairs]
X_a_aligned = X_a[idx_a]  # Hospital A features (aligned)
X_b_aligned = X_b[idx_b]  # Hospital B features (aligned)

# Step 4: VFL training on aligned data
# ... (see Tutorial 10: Vertical FL & PSA)
```

## References

- Schnell, Bachteler & Reiher (2009), *Privacy-preserving record linkage using Bloom filters*, BMC Medical Informatics and Decision Making
- Niedermeyer, Kruse, Kroll & Steinmetzer (2014), *Cryptanalysis of Basic Bloom Filters Used for Privacy Preserving Record Linkage*, Journal of Privacy and Confidentiality
- Christen, Ranbaduge & Vatsalan (2018), *Pattern Mining for Large-Scale Attacks on Bloom Filter-based Record Linkage*, Workshop on Population Informatics
- Randall et al. (2019), *Privacy-preserving record linkage on large real world datasets*, Journal of Biomedical Informatics
- [anonlink](https://github.com/data61/anonlink) — CSIRO Data61, Apache-2.0
- [clkhash](https://github.com/data61/clkhash) — CSIRO Data61, Apache-2.0

## Next Steps

- [Tutorial 10: Vertical FL & PSA](../advanced/10-vertical-fl.md) — use PSA in a full VFL training pipeline
- [DP: Gradient Privacy](dp-gradient-privacy.ipynb) — protect model updates during training
- [PET Reference](../reference/PET_Reference.md) — full technical comparison of all PETs